# Entrenamiento v3 — XGBoost + Grid Search + k-fold CV

En este notebook entrenamos un modelo XGBoost con optimización de hiperparámetros
mediante Grid Search y validación cruzada estratificada (k-fold).

**Experimentos anteriores:**
- Exp 0 (notebook 3): LogReg + TF-IDF → F1-macro val: 0.87
- Exp 1 (notebook 5): LogReg + TF-IDF + features manuales → pendiente

**Este notebook (Exp 2):**
1. TF-IDF + features manuales (misma config)
2. Grid Search con StratifiedKFold (k=5) sobre XGBoost
3. Entrenamiento con mejores hiperparámetros
4. Evaluación en validación
5. Guardar artefactos
6. Registro en MLflow (Experimento 2)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_dataset_artificial"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_ia_artificial"
functions._DATASET_TAGS = {"dataset_type": "artificial", "dataset_source": "eu_ai_act_flagged"}

## 1. Carga de datos

In [3]:
import pandas as pd

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/validation.csv")
test_df = pd.read_csv("data/processed/test.csv")

X_train = train_df["text_final"]
y_train = train_df["etiqueta"]
X_val = val_df["text_final"]
y_val = val_df["etiqueta"]
X_test = test_df["text_final"]
y_test = test_df["etiqueta"]

print(f"Train: {len(X_train)} muestras")
print(f"Validation: {len(X_val)} muestras")
print(f"Test: {len(X_test)} muestras")

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/train.csv'

## 2. TF-IDF + Features manuales

In [ ]:
from functions import crear_tfidf, crear_features_manuales, combinar_features

# TF-IDF (misma config que experimentos anteriores)
tfidf, X_train_tfidf, X_val_tfidf, X_test_tfidf = crear_tfidf(
    X_train, X_val, X_test,
    max_features=5000,
    ngram_range=(1, 2),
)

# Features manuales
feat_train = crear_features_manuales(X_train)
feat_val = crear_features_manuales(X_val)
feat_test = crear_features_manuales(X_test)

# Concatenar
X_train_combined = combinar_features(X_train_tfidf, feat_train)
X_val_combined = combinar_features(X_val_tfidf, feat_val)
X_test_combined = combinar_features(X_test_tfidf, feat_test)

print(f"\nFeatures totales: {X_train_combined.shape[1]} (TF-IDF: {X_train_tfidf.shape[1]} + manuales: {feat_train.shape[1]})")

Vocabulario TF-IDF: 3773 términos
Forma train: (210, 3773)
Forma validation: (45, 3773)
Forma test: (45, 3773)

Features totales: 3780 (TF-IDF: 3773 + manuales: 7)


## 3. Grid Search con k-fold CV

In [ ]:
from functions import grid_search_cv

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0],
}

best_model, best_params, cv_results, le = grid_search_cv(
    X_train_combined, y_train,
    param_grid=param_grid,
    cv=5,
)

Fitting 5 folds for each of 54 candidates, totalling 270 fits

=== Resultados Grid Search (5-fold CV) ===
Mejor F1-macro CV: 0.8369
Mejores parámetros: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}


In [ ]:
# Top 10 combinaciones del Grid Search
print("Top 10 combinaciones de hiperparámetros:\n")
cols = ["rank_test_score", "mean_test_score", "std_test_score", "params"]
print(cv_results[cols].head(10).to_string(index=False))

Top 10 combinaciones de hiperparámetros:

 rank_test_score  mean_test_score  std_test_score                                                                         params
               1         0.836888        0.066636 {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
               2         0.836722        0.066427 {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}
               3         0.833221        0.046831  {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}
               4         0.832614        0.050490  {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}
               5         0.831877        0.058060 {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
               6         0.828535        0.052375 {'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}
               7         0.828230        0.033527  {'le

## 4. Evaluación en validación con el mejor modelo

In [ ]:
from sklearn.metrics import classification_report, f1_score

y_val_pred_enc = best_model.predict(X_val_combined)
y_val_pred = le.inverse_transform(y_val_pred_enc)

print("=== Resultados en VALIDACIÓN (XGBoost — mejor modelo) ===\n")
print(classification_report(y_val, y_val_pred))

f1_val = f1_score(y_val, y_val_pred, average="macro")
acc_val = (y_val_pred == y_val.values).mean()

print(f"F1-score macro (validación): {f1_val:.4f}")
print(f"Accuracy (validación): {acc_val:.4f}")

=== Resultados en VALIDACIÓN (XGBoost — mejor modelo) ===

                 precision    recall  f1-score   support

    alto_riesgo       0.68      0.93      0.79        14
    inaceptable       0.88      0.64      0.74        11
riesgo_limitado       1.00      0.90      0.95        10
  riesgo_minimo       0.78      0.70      0.74        10

       accuracy                           0.80        45
      macro avg       0.83      0.79      0.80        45
   weighted avg       0.82      0.80      0.80        45

F1-score macro (validación): 0.8022
Accuracy (validación): 0.8000


## 5. Comparativa rápida con experimentos anteriores

In [ ]:
from functions import entrenar_modelo_baseline

# Exp 0: baseline
BASELINE_F1 = 0.8698

# Exp 1: LogReg + TF-IDF + features manuales (se calcula aquí directamente)
modelo_v2 = entrenar_modelo_baseline(X_train_combined, y_train, X_val_combined, y_val)
y_val_pred_v2 = modelo_v2.predict(X_val_combined)
EXP1_F1 = f1_score(y_val, y_val_pred_v2, average="macro")

print("\n=== COMPARATIVA VALIDACIÓN ===")
print(f"{'Experimento':<40} {'F1-macro val':>12}")
print("-" * 55)
print(f"{'Exp 0: LogReg + TF-IDF':<40} {BASELINE_F1:>12.4f}")
print(f"{'Exp 1: LogReg + TF-IDF + feat. manuales':<40} {EXP1_F1:>12.4f}")
print(f"{'Exp 2: XGBoost + Grid Search + k-fold':<40} {f1_val:>12.4f}")

=== Resultados en VALIDACIÓN ===

                 precision    recall  f1-score   support

    alto_riesgo       0.72      0.93      0.81        14
    inaceptable       0.89      0.73      0.80        11
riesgo_limitado       1.00      0.90      0.95        10
  riesgo_minimo       0.89      0.80      0.84        10

       accuracy                           0.84        45
      macro avg       0.88      0.84      0.85        45
   weighted avg       0.86      0.84      0.85        45

F1-score macro (validación): 0.8505

=== COMPARATIVA VALIDACIÓN ===
Experimento                              F1-macro val
-------------------------------------------------------
Exp 0: LogReg + TF-IDF                         0.8698
Exp 1: LogReg + TF-IDF + feat. manuales        0.8505
Exp 2: XGBoost + Grid Search + k-fold          0.8022


## 6. Guardar artefactos

In [ ]:
import joblib
import os

output_dir = "model"
os.makedirs(output_dir, exist_ok=True)

# Guardar modelo XGBoost (con LabelEncoder incluido)
path_xgb = os.path.join(output_dir, "modelo_xgboost.joblib")
joblib.dump(best_model, path_xgb)
print(f"Modelo XGBoost guardado en: {path_xgb}")

# Guardar el vectorizador (mismo que los anteriores)
path_tfidf = os.path.join(output_dir, "tfidf_vectorizer.joblib")
joblib.dump(tfidf, path_tfidf)
print(f"Vectorizador guardado en: {path_tfidf}")

# Guardar LabelEncoder por separado
path_le = os.path.join(output_dir, "label_encoder.joblib")
joblib.dump(le, path_le)
print(f"LabelEncoder guardado en: {path_le}")

Modelo XGBoost guardado en: model\modelo_xgboost.joblib
Vectorizador guardado en: model\tfidf_vectorizer.joblib
LabelEncoder guardado en: model\label_encoder.joblib


## 7. Registro en MLflow — Experimento 2

In [ ]:
# MLflow (solo falla esta celda si el servidor no está disponible)
import mlflow
from functions import configure_mlflow, MLFLOW_EXPERIMENT, _DATASET_TAGS

# Guard: verificar que los artefactos de la celda anterior existen
_missing = [v for v in ("path_tfidf", "path_le", "best_model") if v not in dir()]
if _missing:
    print(f"⚠ Variables no definidas: {_missing}. Ejecuta primero la celda de guardado de artefactos.")
else:
    try:
        configure_mlflow()
        mlflow.set_experiment(MLFLOW_EXPERIMENT)

        with mlflow.start_run(run_name="v3_xgboost_gridsearch", tags=_DATASET_TAGS):
            mlflow.log_param("tfidf_max_features",  5000)
            mlflow.log_param("tfidf_ngram_range",   "(1, 2)")
            mlflow.log_param("tfidf_sublinear_tf",  True)
            mlflow.log_param("modelo",              "XGBClassifier")
            for param_name, param_val in best_params.items():
                mlflow.log_param(f"xgb_{param_name}", param_val)
            mlflow.log_param("features_manuales",   list(feat_train.columns))
            mlflow.log_param("total_features",      X_train_combined.shape[1])
            mlflow.log_param("cv_folds",            5)

            mlflow.log_metric("best_cv_f1_macro",   cv_results.iloc[0]["mean_test_score"])
            mlflow.log_metric("val_f1_macro",        f1_val)
            mlflow.log_metric("val_accuracy",        acc_val)
            mlflow.log_metric("baseline_f1_macro",   BASELINE_F1)

            mlflow.log_artifact(path_tfidf, artifact_path="artifacts")
            mlflow.log_artifact(path_le, artifact_path="artifacts")
            mlflow.xgboost.log_model(best_model, "model")

            print("✓ Exp 2 (XGBoost) registrado en MLflow")
            print(f"  Best CV F1-macro: {cv_results.iloc[0]['mean_test_score']:.4f}")
            print(f"  Val F1-macro: {f1_val:.4f}")
            print(f"  Run ID: {mlflow.active_run().info.run_id}")
    except (mlflow.exceptions.MlflowException, ConnectionError, OSError) as e:
        print(f"⚠ MLflow no disponible ({type(e).__name__}): {e}")
    except Exception:
        raise


Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://18.201.64.41/


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\r

✓ Exp 2 (XGBoost) registrado en MLflow
  Best CV F1-macro: 0.8369
  Val F1-macro: 0.8022
  Run ID: 8b87db6eb6074ac89bd62dbe6b0ba8f8


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
